# Revised RL Pipeline: Step-by-Step Notebook

This notebook runs the full revised workflow in stages:
1. Setup and imports
2. Show agent and customer action/state spaces
3. Run data labeling pipeline
4. Identify low-confidence or suspicious labels
5. Prepare and apply gold-label corrections
6. Train persona model
7. Train reward model
8. Train RL pipeline (BC -> CQL-lite -> PPO-style)
9. Run validation suite

Use this when you want visibility and manual checkpoints before training final models.

In [8]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name.lower() == 'simulation' else Path.cwd().resolve()
SIM_ROOT = ROOT / 'Simulation'
SRC_ROOT = SIM_ROOT / 'src'

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
if str(SIM_ROOT) not in sys.path:
    sys.path.insert(0, str(SIM_ROOT))

def checkpoint(msg: str) -> None:
    print(f'[checkpoint] {time.strftime("%H:%M:%S")} | {msg}')

print('ROOT      :', ROOT)
print('SIM_ROOT  :', SIM_ROOT)
print('SRC_ROOT  :', SRC_ROOT)
checkpoint('Notebook setup complete')

ROOT      : B:\College\RL\AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service
SIM_ROOT  : B:\College\RL\AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service\Simulation
SRC_ROOT  : B:\College\RL\AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service\Simulation\src
[checkpoint] 18:44:54 | Notebook setup complete


In [9]:
from simulation_core.config import ACTION_SPACE_7, ACTION_SPACE_8

# Agent action spaces
print('Agent environment actions (7):')
for i, a in enumerate(ACTION_SPACE_7, start=1):
    print(f'  {i}. {a}')

print('\nAgent labeling classes (8):')
for i, a in enumerate(ACTION_SPACE_8, start=1):
    print(f'  {i}. {a}')

# Customer-side behavior/state categories used by revised modules
customer_intents = ['resolution_seeking', 'venting', 'info_seeking', 'escalating']
print('\nCustomer intent categories in environment reset:')
for i, c in enumerate(customer_intents, start=1):
    print(f'  {i}. {c}')

print('\nCustomer state signals predicted/labeled:')
print('  - sentiment_score in [-1, 1]')
print('  - frustration_score in [0, 1]')
print('  - confidence values in [0, 1]')

Agent environment actions (7):
  1. Ask_for_Information
  2. Provide_Solution
  3. Affective_Repair
  4. Escalate_to_Human
  5. Close_with_Feedback
  6. Proactive_Update
  7. Set_Expectation

Agent labeling classes (8):
  1. Ask_for_Information
  2. Provide_Solution
  3. Affective_Repair
  4. Escalate_to_Human
  5. Close_with_Feedback
  6. Proactive_Update
  7. Set_Expectation
  8. Unknown

Customer intent categories in environment reset:
  1. resolution_seeking
  2. venting
  3. info_seeking
  4. escalating

Customer state signals predicted/labeled:
  - sentiment_score in [-1, 1]
  - frustration_score in [0, 1]
  - confidence values in [0, 1]


## Step 1: Run Data + Labeling Pipeline

This generates labeled Twitter/OpenAssistant turn files and a labeling report.

In [ ]:
from simulation_core.data.data_pipeline_v2 import run_pipeline
from simulation_core.labeling import llm_labeler

# Larger run controls.
# Set to None for true full-data processing; 2000/2000 is a strong intermediate scale test.
TWITTER_MAX_THREADS = 2000
OPENASSISTANT_MAX_CONVERSATIONS = 2000
RANDOM_STATE = 42

# Resource controls
MAX_WORKERS = 6
PROGRESS_EVERY = 250
CHECKPOINT_EVERY = 1000

# Resume controls for expensive labeling stage.
# False forces a fresh run so new sampling params are applied.
RESUME_LABELING_IF_EXISTS = False

# Keep False for production-quality labels; True is only for debugging.
FORCE_HEURISTIC_LABELING = False
if FORCE_HEURISTIC_LABELING:
    llm_labeler.chat = None

# LLM preflight: fail early instead of wasting time on a long run.
if llm_labeler.chat is None and not FORCE_HEURISTIC_LABELING:
    raise RuntimeError(
        'Ollama chat is unavailable. Start Ollama and ensure the model is installed before running Step 1.'
    )
_probe = llm_labeler.label_action_with_llm('Please share your account number so I can check this issue.')
print('LLM preflight action label:', _probe.get('action_label'))

checkpoint('Starting Step 1: data + labeling pipeline')
t0 = time.time()
outputs = run_pipeline(
    twitter_max_threads=TWITTER_MAX_THREADS,
    openassistant_max_conversations=OPENASSISTANT_MAX_CONVERSATIONS,
    random_state=RANDOM_STATE,
    max_workers=MAX_WORKERS,
    progress_every=PROGRESS_EVERY,
    resume_if_exists=RESUME_LABELING_IF_EXISTS,
    checkpoint_every=CHECKPOINT_EVERY,
)
dt = time.time() - t0

outputs = {k: Path(v) for k, v in outputs.items()}
for k, v in outputs.items():
    print(f'{k}: {v}')
    print('exists:', v.exists())

print('---')
print('Sampling config:')
print('twitter_max_threads:', TWITTER_MAX_THREADS)
print('openassistant_max_conversations:', OPENASSISTANT_MAX_CONVERSATIONS)
print('random_state:', RANDOM_STATE)
print('max_workers:', MAX_WORKERS)
print('progress_every:', PROGRESS_EVERY)
print('checkpoint_every:', CHECKPOINT_EVERY)
print('resume_labeling_if_exists:', RESUME_LABELING_IF_EXISTS)
print('force_heuristic_labeling:', FORCE_HEURISTIC_LABELING)
print('step1_elapsed_sec:', round(dt, 2))
checkpoint('Step 1 completed')

[checkpoint] 18:46:00 | Starting Step 1: data + labeling pipeline
[run_pipeline] Loading source turns...
[run_pipeline] Twitter turns: 22
[run_pipeline] OpenAssistant turns: 37
[run_pipeline] Annotating turns...
[annotate_turns] 22/22 rows labeled
[annotate_turns] checkpoint saved: B:\College\RL\AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service\Simulation\artifacts\revised_pipeline\labeled_data\twitter_labeled.partial.csv (22/22)
[annotate_turns] 37/37 rows labeled
[annotate_turns] checkpoint saved: B:\College\RL\AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service\Simulation\artifacts\revised_pipeline\labeled_data\openassistant_labeled.partial.csv (37/37)
[run_pipeline] Wrote: B:\College\RL\AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service\Simulation\artifacts\revised_pipeline\labeled_data\twitter_labeled.csv
[run_pipeline] Wrote: B:\College\RL\AdaptiveBandit--Contextual-Bandits-fo

In [ ]:
checkpoint('Loading labeled outputs into memory')
twitter_labeled = pd.read_csv(outputs['twitter_labeled'])
openassistant_labeled = pd.read_csv(outputs['openassistant_labeled'])
labeled_all = pd.concat([twitter_labeled, openassistant_labeled], ignore_index=True)

print('Twitter rows      :', len(twitter_labeled))
print('OpenAssistant rows:', len(openassistant_labeled))
print('Total rows        :', len(labeled_all))

# Quick integrity checks
required_cols = ['conv_id', 'turn_id', 'speaker_role', 'text', 'tier', 'tier_confidence']
missing_cols = [c for c in required_cols if c not in labeled_all.columns]
print('Missing required columns:', missing_cols if missing_cols else 'None')
print('Null text rows:', int(labeled_all['text'].isna().sum()))

labeled_all.head(3)

## Step 2: Find Suspect Labels for Human Review

This surfaces likely LLM mistakes (especially low confidence) for gold correction.

In [ ]:
checkpoint('Starting Step 2: suspect-label mining')
t0 = time.time()

review_dir = SIM_ROOT / 'artifacts' / 'revised_pipeline' / 'gold_review'
review_dir.mkdir(parents=True, exist_ok=True)

agent_turns = labeled_all[labeled_all['speaker_role'] == 'agent'].copy()
agent_turns['action_confidence'] = pd.to_numeric(agent_turns['action_confidence'], errors='coerce').fillna(0.0)

# Priority sample: lowest-confidence predictions first
review_candidates = agent_turns.sort_values('action_confidence', ascending=True).head(300).copy()
review_cols = [
    'conv_id', 'turn_id', 'domain_source', 'speaker_role', 'text',
    'action_label', 'action_confidence', 'annotator_confidence', 'tier', 'tier_confidence'
 ]
review_candidates = review_candidates[[c for c in review_cols if c in review_candidates.columns]]

review_csv = review_dir / 'action_review_candidates.csv'
review_candidates.to_csv(review_csv, index=False)

print('Saved review file:', review_csv)
print('exists:', review_csv.exists())
print('Rows for manual review:', len(review_candidates))
print('Action confidence min/median/max:',
      float(review_candidates['action_confidence'].min()) if len(review_candidates) else None,
      float(review_candidates['action_confidence'].median()) if len(review_candidates) else None,
      float(review_candidates['action_confidence'].max()) if len(review_candidates) else None)
print('step2_elapsed_sec:', round(time.time() - t0, 2))
checkpoint('Step 2 completed')

review_candidates.head(10)

## Step 3: Add Gold Corrections

1. Open the CSV saved above and add a new column named `gold_action_label`.
2. Fill it only when the model label is wrong.
3. Save as `action_gold_labels.csv` in the same folder.

Tip: you can also use the local UI tool at `Simulation/ui/gold_annotation_tool.html` for structured annotation workflows.

In [ ]:
checkpoint('Starting Step 3: apply gold corrections if available')
t0 = time.time()

gold_path = review_dir / 'action_gold_labels.csv'
if not gold_path.exists():
    print('No gold file found yet:', gold_path)
    print('Create it from action_review_candidates.csv with a gold_action_label column, then re-run this cell.')
else:
    gold_df = pd.read_csv(gold_path)
    required = {'conv_id', 'turn_id', 'gold_action_label'}
    missing = required - set(gold_df.columns)
    if missing:
        raise ValueError(f'Missing required columns in gold file: {missing}')

    gold_df = gold_df.dropna(subset=['gold_action_label']).copy()
    gold_df['turn_id'] = pd.to_numeric(gold_df['turn_id'], errors='coerce').astype('Int64')

    merged = labeled_all.copy()
    merged['turn_id'] = pd.to_numeric(merged['turn_id'], errors='coerce').astype('Int64')

    key_cols = ['conv_id', 'turn_id']
    merged = merged.merge(gold_df[key_cols + ['gold_action_label']], on=key_cols, how='left')

    before = merged['action_label'].copy()
    merged['action_label'] = np.where(
        merged['gold_action_label'].notna(),
        merged['gold_action_label'],
        merged['action_label']
    )

    n_changed = int((before != merged['action_label']).fillna(False).sum())
    corrected_path = review_dir / 'labeled_with_gold_corrections.csv'
    merged.to_csv(corrected_path, index=False)

    print('Gold rows provided:', len(gold_df))
    print('Applied corrections:', n_changed)
    print('Saved corrected dataset:', corrected_path)
    print('exists:', corrected_path.exists())

print('step3_elapsed_sec:', round(time.time() - t0, 2))
checkpoint('Step 3 completed')

## Step 4: Choose Training Dataset

If gold-corrected labels exist, use them. Otherwise use original labeled data.

In [ ]:
checkpoint('Starting Step 4: training dataset selection')
t0 = time.time()

corrected_path = review_dir / 'labeled_with_gold_corrections.csv'
if corrected_path.exists():
    train_df = pd.read_csv(corrected_path)
    source_name = 'gold-corrected'
else:
    train_df = labeled_all.copy()
    source_name = 'original-labeled'

print('Training data source:', source_name)
print('Rows:', len(train_df))
print('Unique conversations:', int(train_df['conv_id'].nunique()) if 'conv_id' in train_df.columns else 'N/A')
print('Agent rows:', int((train_df['speaker_role'] == 'agent').sum()) if 'speaker_role' in train_df.columns else 'N/A')
print('Customer rows:', int((train_df['speaker_role'] == 'customer').sum()) if 'speaker_role' in train_df.columns else 'N/A')
print('step4_elapsed_sec:', round(time.time() - t0, 2))
checkpoint('Step 4 completed')

## Step 5: Train Persona, Reward, and RL Models

In [ ]:
from simulation_core.persona.persona_model import train_persona_cvae
from simulation_core.rewards.reward_model import train_reward_model
from simulation_core.training.training_pipeline import run_training_pipeline

artifacts_root = SIM_ROOT / 'artifacts' / 'revised_pipeline'
persona_dir = artifacts_root / 'persona'
reward_dir = artifacts_root / 'reward'
training_dir = artifacts_root / 'training'

# Training budget controls for resource-aware execution.
BC_EPOCHS = 20
CQL_STEPS = 100
PPO_STEPS = 200

# Resume expensive stages from existing artifacts when possible.
RESUME_PERSONA = True
RESUME_REWARD = True
RESUME_POLICY = True

checkpoint('Starting Step 5: model training stack')

persona_json = persona_dir / 'persona_validation.json'
persona_ckpt = persona_dir / 'persona_cvae.pt'
if RESUME_PERSONA and persona_json.exists() and persona_ckpt.exists():
    checkpoint('Reusing existing persona artifacts')
    persona_metrics = json.loads(persona_json.read_text(encoding='utf-8'))
else:
    checkpoint('Training persona model')
    t0 = time.time()
    persona_metrics = train_persona_cvae(train_df, persona_dir)
    print('persona_elapsed_sec:', round(time.time() - t0, 2))

reward_json = reward_dir / 'reward_validation.json'
reward_weights = reward_dir / 'reward_weights.npy'
if RESUME_REWARD and reward_json.exists() and reward_weights.exists():
    checkpoint('Reusing existing reward artifacts')
    reward_metrics = json.loads(reward_json.read_text(encoding='utf-8'))
else:
    checkpoint('Training reward model')
    t1 = time.time()
    reward_metrics = train_reward_model(train_df, reward_dir)
    print('reward_elapsed_sec:', round(time.time() - t1, 2))

training_json = training_dir / 'training_summary.json'
if RESUME_POLICY and training_json.exists() and reward_weights.exists():
    checkpoint('Reusing existing policy artifacts')
    training_metrics = json.loads(training_json.read_text(encoding='utf-8'))
else:
    checkpoint('Training policy pipeline')
    t2 = time.time()
    training_metrics = run_training_pipeline(
        train_df,
        reward_weights,
        training_dir,
        bc_epochs=BC_EPOCHS,
        cql_steps=CQL_STEPS,
        ppo_steps=PPO_STEPS,
     )
    print('training_elapsed_sec:', round(time.time() - t2, 2))

print('Persona metrics:', json.dumps(persona_metrics, indent=2))
print('Reward metrics :', json.dumps(reward_metrics, indent=2))
print('Training done. Summary keys:', list(training_metrics.keys()))
checkpoint('Step 5 completed')

## Step 6: Run Validation Suite

In [ ]:
from simulation_core.evaluation.validation_suite import ValidationSuite

# Validation budget controls.
BANDIT_STEPS = 800

# Resume expensive validation stage if prior result exists.
RESUME_VALIDATION = True

checkpoint('Starting Step 6: validation suite')
t0 = time.time()
suite = ValidationSuite(real_df=train_df)
validation_dir = artifacts_root / 'validation'
validation_json = validation_dir / 'validation_results.json'

if RESUME_VALIDATION and validation_json.exists():
    checkpoint('Reusing existing validation artifacts')
    validation_results = json.loads(validation_json.read_text(encoding='utf-8'))
else:
    validation_results = suite.run(simulated_df=train_df.copy(), out_dir=validation_dir, bandit_steps=BANDIT_STEPS)

print('validation_elapsed_sec:', round(time.time() - t0, 2))
print(json.dumps(validation_results, indent=2))
print('Saved:', validation_json)
print('exists:', validation_json.exists())
checkpoint('Step 6 completed')

In [ ]:
checkpoint('Writing final run summary')
t0 = time.time()

final_summary = {
    'data_source': source_name,
    'pipeline_outputs': {k: str(v) for k, v in outputs.items()},
    'persona': persona_metrics if 'persona_metrics' in globals() else {},
    'reward': reward_metrics if 'reward_metrics' in globals() else {},
    'validation': validation_results if 'validation_results' in globals() else {},
}

summary_path = artifacts_root / 'notebook_run_summary.json'
summary_path.write_text(json.dumps(final_summary, indent=2), encoding='utf-8')
print('Saved notebook summary:', summary_path)
print('exists:', summary_path.exists())
print('size_bytes:', summary_path.stat().st_size if summary_path.exists() else 0)
print('step7_elapsed_sec:', round(time.time() - t0, 2))
checkpoint('Notebook run completed')

## Notes

- If LLM labels are poor, do one or more gold-correction cycles before re-training.
- The most important human-check point is low-confidence agent action labels.
- You can repeat Steps 2-5 iteratively to improve model quality.